# NJ Dynamic Panel Baseline: Daily Cases at Horizons 1, 2 and 3

This experiment starts with daily recorded-onset case counts. It fits separate direct forecasts for days 1, 2 and 3, then saves predictions, MAE, RMSE, training logs, coefficients and experiment settings. It does not generate outbreak labels or calculate PR-AUC / warning lead time.

Dependencies: numpy and pandas. Read the compressed feature panel directly; no decryption or Excel file is needed. The notebook is self-contained. Delivered cells contain no real-data execution output.


## Starting design (editable, not a selected optimum)

| Setting | Starting value |
|---|---|
| Training window | 90 calendar origin dates per region |
| Parameter refit | Every 7 forecast-origin dates, restarted at each evaluation split |
| Input updates | Daily, using observed history through the current origin date |
| Horizons | Three direct models: t+1, t+2, t+3, not a 3-day cumulative total |
| Validation period | Origins in 2021, keeping every horizon target within 2021 |
| Initial test period | Origins in 2022–2023, keeping every target within that period |
| Main baseline cohort | All panel regions with complete historical inputs |
| Pressure comparison | Same eligible regions for both training and scoring in the paired comparison |
| Metrics | MAE, RMSE; additionally bias and observed mean for interpretation |
| Count output | Clip negative forecasts to zero, do not round; preserve raw forecasts |

For common horizon origins, the last three origin dates are excluded at each split's end. The first forecasts begin January 1, 2021 and January 1, 2022. This does not estimate the final best training window. Compare 30/60/90/180-day choices on the same validation dates later, without choosing based on test scores.


## Model and boundary rules

For each h=1,2,3, fit a pooled dynamic panel with region-specific fixed intercepts and shared coefficients for `cases_today`, `cases_lag_1`, `cases_lag_7`, plus target-day weekday dummies. Monday is the reference weekday. Parameters are estimated with within-region OLS. This is a predictive fixed-effects baseline, not GMM or causal estimation. Population, density and all other static columns are not separately included with full region fixed effects.

At a refit date t, the last training origin for horizon h is t-h, so its outcome occurs at t. The preceding 90 origin dates form that model's window. No training target occurs after t. Between weekly refits, parameters stay fixed but current case inputs update daily. During test, earlier test-period observations may enter subsequent rolling training once they have occurred. This evaluates a periodically retrained system, not one model frozen for two years.

Simple baselines are `future = today` and the observed count from the same weekday one week before each target date. No future observed day-1/day-2 values are used to predict day 3.

The optional pressure model adds `nj_covid_pressure_7d`. Its counterpart without pressure is refitted on the SAME source-coverage-eligible regions. Main all-region results and paired-cohort results are stored separately; do not compare different cohorts' errors as evidence of pressure benefit.


## Data interpretation

The outcome remains `cases_today` from the final recorded-onset panel. Unknown onset dates were excluded upstream, and zero-record days are zero under the user's confirmed rule. This is retrospective onset-date evaluation, not a simulation of when cases first became reportable. The 2021 geographic and demographic inputs are static; ZIP-to-ZCTA correspondence is still unvalidated. Interpret optional pressure results subject to these assumptions.

Missing onset is especially common in 2023. Metrics are therefore also saved separately by target year. Lower error in a low-count year is not by itself better epidemiological performance.


## 1. Define the model and evaluation functions

In [8]:
"""Retrospective fixed-effects OLS baseline for direct 1/2/3-day onset-count forecasts.

Dependencies: numpy, pandas. No GMM, causal estimation or outbreak classification.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import numpy as np
import pandas as pd

DEFAULT_SOURCE = Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\nj_daily_feature_outputs\nj_daily_features.csv.gz')
DEFAULT_OUTPUT = DEFAULT_SOURCE.parent.parent / 'dynamic_panel_nj_outputs'


def fit_within_ols(x, y):
    """x[region, training day, feature], y[region, training day]."""
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm, ym = x.mean(axis=1), y.mean(axis=1)
    xw, yw = x-xm[:,None,:], y-ym[:,None]
    scale = np.sqrt(np.mean(xw*xw, axis=(0,1)))
    scale = np.where(scale>1e-12,scale,1)
    design = (xw/scale).reshape(-1,x.shape[2])
    bscaled, _, rank, _ = np.linalg.lstsq(design,yw.reshape(-1),rcond=None)
    beta = bscaled/scale
    alpha = ym-xm@beta
    assert np.isfinite(beta).all() and np.isfinite(alpha).all()
    return beta,alpha,int(rank)


def metric_rows(frame, columns, group_columns):
    """Evaluate count predictions; each model's usable observations are explicit."""
    rows=[]
    for key,part in frame.groupby(group_columns,sort=True):
        key = key if isinstance(key,tuple) else (key,)
        for model,column in columns.items():
            good = part[column].notna() & part.actual_cases.notna()
            q=part.loc[good]
            if q.empty:
                continue
            errors=q[column].to_numpy()-q.actual_cases.to_numpy()
            rows.append({**dict(zip(group_columns,key)), 'model':model,
                'rows':len(q),'regions':q.zip.nunique(),'forecast_dates':q.date.nunique(),
                'MAE':float(np.mean(np.abs(errors))),'RMSE':float(np.sqrt(np.mean(errors**2))),
                'mean_error_prediction_minus_actual':float(np.mean(errors)),
                'mean_actual':float(q.actual_cases.mean()),
                'target_start':str(q.target_date.min().date()),'target_end':str(q.target_date.max().date())})
    return pd.DataFrame(rows)


def run_experiment(source=DEFAULT_SOURCE, output=DEFAULT_OUTPUT, window=90, retrain_every=7,
                   validation_start='2021-01-01', validation_end='2021-12-31',
                   test_start='2022-01-01', test_end='2023-12-31', include_pressure=True):
    assert window>=14 and retrain_every>=1
    source,output=Path(source),Path(output)
    assert source.is_file(), f'Cannot find {source}'
    output.mkdir(parents=True,exist_ok=True)
    run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_folder=output/f'window_{window}d_{run_id}'
    run_folder.mkdir()
    use=['zip','date','cases_today','cases_lag_1','cases_lag_7','population','valid_population_denominator']
    if include_pressure:
        use+=['nj_covid_pressure_7d','pressure_source_weight_coverage_nj_7d']
    panel=pd.read_csv(source,usecols=use,dtype={'zip':'string'},parse_dates=['date'])
    panel.zip=panel.zip.str.strip().str.zfill(5)
    assert panel.zip.str.fullmatch(r'[0-9]{5}').all() and panel.date.notna().all()
    assert not panel.duplicated(['zip','date']).any()
    ids=sorted(panel.zip.unique()); dates=pd.date_range(panel.date.min(),panel.date.max())
    assert len(panel)==len(ids)*len(dates)
    def matrix(c):
        return panel.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(dtype=float)
    counts=matrix('cases_today'); lag1=matrix('cases_lag_1'); lag7=matrix('cases_lag_7')
    assert np.isfinite(counts).all() and (counts>=0).all()
    assert np.array_equal(lag1[:,1:],counts[:,:-1])
    assert np.array_equal(lag7[:,7:],counts[:,:-7])
    n,T=counts.shape
    x_base=np.stack([counts,lag1,lag7],axis=2)
    pressure=matrix('nj_covid_pressure_7d') if include_pressure else None
    if include_pressure:
        # Eligibility is determined on the first full trailing-window day, using
        # static network/population availability, not later case outcomes.
        complete_coverage=matrix('pressure_source_weight_coverage_nj_7d')
        pressure_eligible=np.isfinite(pressure[:,6]) & np.isclose(complete_coverage[:,6],1,rtol=0,atol=1e-10)
        if pressure_eligible.any():
            assert np.isfinite(pressure[pressure_eligible,6:]).all(), 'Source coverage changes: explicitly revise the comparison cohort.'
    else:
        pressure_eligible=np.zeros(n,dtype=bool)
    horizons=[1,2,3]; max_h=max(horizons)
    vs,ve,ts,te=map(pd.Timestamp,[validation_start,validation_end,test_start,test_end])
    assert dates.min()<vs<=ve<ts<=te<=dates.max()
    config={'run_id':run_id,'training_window_calendar_days':window,'retrain_every_forecast_days':retrain_every,
        'horizons':horizons,'validation_target_boundaries':[str(vs.date()),str(ve.date())],
        'test_target_boundaries':[str(ts.date()),str(te.date())],
        'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'model':'Direct horizon-specific dynamic panel with region fixed effects, estimated by within OLS.',
        'features':['cases_today','cases_lag_1','cases_lag_7','target_weekday (Monday reference, six dummies)'],
        'pressure_variant':'Same model + nj_covid_pressure_7d on fixed source-coverage-eligible cohort' if include_pressure else 'Disabled',
        'outcome':'Recorded-onset rows after exact deduplication. Missing onset excluded, absent ZIP/date records zero as confirmed by user.',
        'evaluation':'MAE/RMSE on nonnegative-clipped count forecasts; raw predictions retained. No rounding to integers.',
        'training_boundary':'For fit date t and horizon h, 90 (or configured W) origin dates end at t-h; all training labels <= t.',
        'weekly_policy':'Fixed-effects coefficients refit weekly; each day uses newly observed history. Refit schedule restarts at each split.',
        'test_policy':'Fixed 90-day starting specification, not a validation-selected optimum. During test, earlier test observations become training history after their dates occur; future labels never enter fitting.',
        'geography':'ZIP/ZCTA mapping remains unvalidated; pressure conditional on supplied static population/geographic assumptions.',
        'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__},
        'limitations':['Retrospective onset-date evaluation, not real-time reporting simulation.','Fixed-effects lagged-outcome OLS is a predictive baseline, not causal inference or a GMM estimator.','Onset completeness changes substantially by year; inspect year-specific metrics.','No outbreak labels, PR-AUC, warning lead time or Moran diagnostics computed in this experiment.']}
    (run_folder/'experiment_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    pd.DataFrame({'zip':ids,'pressure_comparison_eligible':pressure_eligible}).to_csv(run_folder/'region_cohort.csv',index=False)
    records=[]; fit_log=[]; coefficients=[]
    # Tests compare within transformation with an explicit region-dummy OLS fit.
    rng=np.random.default_rng(21)
    sx=rng.normal(size=(3,20,2)); sy=np.array([1.,2.,3.])[:,None]+sx@np.array([.4,-.2])+rng.normal(scale=.1,size=(3,20))
    sb,sa,_=fit_within_ols(sx,sy)
    dummy=np.repeat(np.eye(3),20,axis=0)
    direct=np.linalg.lstsq(np.column_stack([dummy,sx.reshape(-1,2)]),sy.reshape(-1),rcond=None)[0]
    assert np.allclose(sa,direct[:3]) and np.allclose(sb,direct[3:])
    print(f'Regions: {n}; pressure comparison regions: {pressure_eligible.sum()}; window: {window} days.',flush=True)
    for h in horizons:
        target_weekdays=(dates+pd.Timedelta(days=h)).dayofweek.to_numpy()
        weekday=np.eye(7)[target_weekdays,1:]
        x=np.concatenate([x_base,np.broadcast_to(weekday,(n,T,6))],axis=2)
        names=['cases_today','cases_lag_1','cases_lag_7']+[f'target_weekday_{i}' for i in range(2,8)]
        for split,start,end in [('validation',vs,ve),('test',ts,te)]:
            # A common origin set across h=1,2,3 keeps all targets within the split.
            origins=np.flatnonzero((dates>=start) & ((dates+pd.Timedelta(days=max_h))<=end))
            assert len(origins)>0
            for block_start in range(0,len(origins),retrain_every):
                block=origins[block_start:block_start+retrain_every]
                fit_i=int(block[0]); train_last=fit_i-h; train_first=train_last-window+1
                assert train_first>=7, 'Not enough pre-validation history for this window.'
                train=np.arange(train_first,train_last+1)
                assert len(train)==window and (train+h).max()<=fit_i
                assert (dates[train]+pd.Timedelta(days=h)).max()<=dates[fit_i]
                y=counts[:,train+h]
                beta,alpha,rank=fit_within_ols(x[:,train,:],y)
                base_raw=np.einsum('ntp,p->nt',x[:,block,:],beta)+alpha[:,None]
                pressure_raw=np.full_like(base_raw,np.nan)
                matched_base_raw=np.full_like(base_raw,np.nan)
                models=[('Dynamic panel FE',beta,alpha,rank,n,names)]
                if include_pressure and pressure_eligible.any():
                    bm,am,rm=fit_within_ols(x[pressure_eligible][:,train,:],y[pressure_eligible])
                    matched_base_raw[pressure_eligible]=np.einsum('ntp,p->nt',x[pressure_eligible][:,block,:],bm)+am[:,None]
                    xp=np.concatenate([x[pressure_eligible],pressure[pressure_eligible,:,None]],axis=2)
                    bp,ap,rp=fit_within_ols(xp[:,train,:],y[pressure_eligible])
                    pressure_raw[pressure_eligible]=np.einsum('ntp,p->nt',xp[:,block,:],bp)+ap[:,None]
                    models.append(('Dynamic panel FE + pressure',bp,ap,rp,int(pressure_eligible.sum()),names+['nj_covid_pressure_7d']))
                    models.append(('Dynamic panel FE (matched cohort)',bm,am,rm,int(pressure_eligible.sum()),names))
                for name,b,a,r,num_regions,feature_names in models:
                    fit_log.append({'split':split,'horizon':h,'model':name,'fit_date':dates[fit_i],
                        'training_origin_start':dates[train_first],'training_origin_end':dates[train_last],
                        'latest_training_target':dates[train_last+h],
                        'prediction_origin_start':dates[block[0]],'prediction_origin_end':dates[block[-1]],
                        'training_dates':window,'training_rows':window*num_regions,'regions':num_regions,
                        'within_design_rank':r,'features':len(b),'negative_raw_predictions':int((base_raw<0).sum()) if name=='Dynamic panel FE' else int(((matched_base_raw if name=='Dynamic panel FE (matched cohort)' else pressure_raw)[pressure_eligible]<0).sum())})
                    for feature,value in zip(feature_names,b):
                        coefficients.append({'split':split,'horizon':h,'model':name,'fit_date':dates[fit_i],'feature':feature,'coefficient':float(value)})
                seasonal=counts[:,block+h-7]
                L=len(block)
                records.append(pd.DataFrame({'split':split,'zip':np.repeat(ids,L),
                    'date':np.tile(dates[block].to_numpy(),n),'target_date':np.tile(dates[block+h].to_numpy(),n),
                    'horizon':h,'fit_date':dates[fit_i],
                    'actual_cases':counts[:,block+h].reshape(-1),
                    'persistence_prediction':counts[:,block].reshape(-1),
                    'seasonal_naive_prediction':seasonal.reshape(-1),
                    'panel_prediction_raw':base_raw.reshape(-1),
                    'panel_prediction':np.maximum(base_raw,0).reshape(-1),
                    'matched_panel_prediction_raw':matched_base_raw.reshape(-1),
                    'matched_panel_prediction':np.maximum(matched_base_raw,0).reshape(-1),
                    'pressure_prediction_raw':pressure_raw.reshape(-1),
                    'pressure_prediction':np.maximum(pressure_raw,0).reshape(-1),
                    'pressure_comparison_eligible':np.repeat(pressure_eligible,L)}))
                if block_start%(retrain_every*13)==0:
                    print(f'h={h}, {split}, forecast block begins {dates[fit_i].date()}',flush=True)
    predictions=pd.concat(records,ignore_index=True)
    assert not predictions.duplicated(['zip','date','horizon']).any()
    assert (predictions.fit_date<=predictions.date).all()
    assert (predictions.target_date==predictions.date+pd.to_timedelta(predictions.horizon,unit='D')).all()
    logs=pd.DataFrame(fit_log)
    assert logs.latest_training_target.le(logs.fit_date).all()
    base_columns={'Dynamic panel FE':'panel_prediction','Persistence: future = today':'persistence_prediction','Seasonal naive: same weekday last week':'seasonal_naive_prediction'}
    metrics=metric_rows(predictions,base_columns,['split','horizon'])
    yearly=metric_rows(predictions.assign(target_year=predictions.target_date.dt.year),base_columns,['split','target_year','horizon'])
    by_region=metric_rows(predictions,base_columns,['split','zip','horizon'])
    metrics['cohort']='all_panel_regions'
    matched=pd.DataFrame()
    if include_pressure and pressure_eligible.any():
        # Every row/model in this table is scored on exactly the same observations.
        pair=predictions.loc[predictions.pressure_comparison_eligible].copy()
        assert pair.pressure_prediction.notna().all()
        matched_columns={**base_columns,'Dynamic panel FE':'matched_panel_prediction','Dynamic panel FE + pressure':'pressure_prediction'}
        matched=metric_rows(pair,matched_columns,['split','horizon'])
        matched['cohort']='pressure_eligible_regions_only'
        assert matched.groupby(['split','horizon']).rows.nunique().eq(1).all()
        matched.to_csv(run_folder/'metrics_matched_pressure_comparison.csv',index=False)
    metrics.to_csv(run_folder/'metrics_by_horizon.csv',index=False)
    yearly.to_csv(run_folder/'metrics_by_year.csv',index=False)
    by_region.to_csv(run_folder/'metrics_by_region.csv',index=False)
    logs.to_csv(run_folder/'training_log.csv',index=False)
    pd.DataFrame(coefficients).to_csv(run_folder/'feature_coefficients.csv',index=False)
    predictions.head(100).to_csv(run_folder/'prediction_preview.csv',index=False)
    print('Saving predictions and experiment records.',flush=True)
    predictions.to_csv(run_folder/'predictions.csv.gz',index=False,compression='gzip')
    details={'run_id':run_id,'rows_predictions':len(predictions),'regions':n,'pressure_comparison_regions':int(pressure_eligible.sum()),
        'model_fits':len(logs),'all_checks_passed':True,'outputs':str(run_folder.resolve()),
        'forecast_origin_dates_by_split':{k:int(g.date.nunique()) for k,g in predictions.groupby('split')},
        'selection_status':'90-day (or configured) starting experiment. Not a validation-selected best window.'}
    (run_folder/'run_summary.json').write_text(json.dumps(details,indent=2),encoding='utf-8')
    print(metrics.to_string(index=False),flush=True)
    print('Saved to:',run_folder.resolve(),flush=True)
    return predictions,metrics,matched,run_folder




## 2. Configure and run

Open this notebook from your local `Weikun` directory, or change SOURCE to the feature file's absolute path inside nj_daily_feature_outputs. Output is saved in Weikun/dynamic_panel_nj_outputs. Each run gets a new folder so previous results are retained. The run may take several minutes, including export of all predictions.


In [9]:
SOURCE = Path.cwd() / "nj_daily_feature_outputs" / "nj_daily_features.csv.gz"
assert SOURCE.is_file(), "Set SOURCE to your nj_daily_features.csv.gz file."
OUTPUT = Path.cwd() / "dynamic_panel_nj_outputs"

TRAIN_WINDOW = 90
RETRAIN_EVERY = 7
VALIDATION_START = "2021-01-01"
VALIDATION_END = "2021-12-31"
TEST_START = "2022-01-01"
TEST_END = "2023-12-31"

predictions, metrics, matched_metrics, RUN_FOLDER = run_experiment(
    source=SOURCE,
    output=OUTPUT,
    window=TRAIN_WINDOW,
    retrain_every=RETRAIN_EVERY,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
    include_pressure=True,
)


Regions: 598; pressure comparison regions: 590; window: 90 days.
h=1, validation, forecast block begins 2021-01-01
h=1, validation, forecast block begins 2021-04-02
h=1, validation, forecast block begins 2021-07-02
h=1, validation, forecast block begins 2021-10-01
h=1, test, forecast block begins 2022-01-01
h=1, test, forecast block begins 2022-04-02
h=1, test, forecast block begins 2022-07-02
h=1, test, forecast block begins 2022-10-01
h=1, test, forecast block begins 2022-12-31
h=1, test, forecast block begins 2023-04-01
h=1, test, forecast block begins 2023-07-01
h=1, test, forecast block begins 2023-09-30
h=2, validation, forecast block begins 2021-01-01
h=2, validation, forecast block begins 2021-04-02
h=2, validation, forecast block begins 2021-07-02
h=2, validation, forecast block begins 2021-10-01
h=2, test, forecast block begins 2022-01-01
h=2, test, forecast block begins 2022-04-02
h=2, test, forecast block begins 2022-07-02
h=2, test, forecast block begins 2022-10-01
h=2, te

## 3. Review MAE and RMSE

Lower MAE/RMSE is better. Each horizon is scored separately. The all-region table is the main initial baseline result. Compare the two dynamic-panel variants using the paired-cohort table only. Use validation results for later window/model choices. Test results are the initial fixed-configuration benchmark, not evidence that a window has been selected optimally.


In [10]:
display(metrics[["split", "horizon", "model", "rows", "regions", "MAE", "RMSE", "mean_actual"]].round({"MAE":4,"RMSE":4,"mean_actual":4}))
if not matched_metrics.empty:
    display(matched_metrics[["split", "horizon", "model", "rows", "regions", "MAE", "RMSE"]].round({"MAE":4,"RMSE":4}))
display(pd.read_csv(RUN_FOLDER / "metrics_by_year.csv"))
print("Results saved to:", RUN_FOLDER.resolve())


,split,horizon,model,rows,regions,MAE,RMSE,mean_actual
0,test,1,Dynamic panel FE,434746,598,0.5704,1.0868,0.7768
1,test,1,Persistence: future = today,434746,598,0.6144,1.3141,0.7768
2,test,1,Seasonal naive: same weekday last week,434746,598,0.6611,1.4957,0.7768
3,test,2,Dynamic panel FE,434746,598,0.5898,1.1303,0.7695
4,test,2,Persistence: future = today,434746,598,0.6282,1.3584,0.7695
5,test,2,Seasonal naive: same weekday last week,434746,598,0.6546,1.4640,0.7695
6,test,3,Dynamic panel FE,434746,598,0.6026,1.1612,0.7600
7,test,3,Persistence: future = today,434746,598,0.6325,1.3752,0.7600
8,test,3,Seasonal naive: same weekday last week,434746,598,0.6478,1.4296,0.7600
9,validation,1,Dynamic panel FE,216476,598,1.1100,1.9039,1.9866


,split,horizon,model,rows,regions,MAE,RMSE
0,test,1,Dynamic panel FE,428930,590,0.5775,1.0941
1,test,1,Persistence: future = today,428930,590,0.6226,1.3229
2,test,1,Seasonal naive: same weekday last week,428930,590,0.6700,1.5058
3,test,1,Dynamic panel FE + pressure,428930,590,0.5501,1.0756
4,test,2,Dynamic panel FE,428930,590,0.5970,1.1379
5,test,2,Persistence: future = today,428930,590,0.6366,1.3675
6,test,2,Seasonal naive: same weekday last week,428930,590,0.6633,1.4738
7,test,2,Dynamic panel FE + pressure,428930,590,0.5656,1.1160
8,test,3,Dynamic panel FE,428930,590,0.6099,1.1690
9,test,3,Persistence: future = today,428930,590,0.6410,1.3845


,split,target_year,horizon,model,rows,regions,forecast_dates,MAE,RMSE,mean_error_prediction_minus_actual,mean_actual,target_start,target_end
0,test,2022,1,Dynamic panel FE,217672,598,364,0.920798,1.454246,0.134664,1.384358,2022-01-02,2022-12-31
1,test,2022,1,Persistence: future = today,217672,598,364,1.028373,1.754561,0.012542,1.384358,2022-01-02,2022-12-31
2,test,2022,1,Seasonal naive: same weekday last week,217672,598,364,1.113134,2.018463,0.124233,1.384358,2022-01-02,2022-12-31
3,test,2022,2,Dynamic panel FE,217074,598,363,0.956940,1.518101,0.178434,1.373283,2022-01-03,2022-12-31
4,test,2022,2,Persistence: future = today,217074,598,363,1.055520,1.818410,0.024941,1.373283,2022-01-03,2022-12-31
5,test,2022,2,Seasonal naive: same weekday last week,217074,598,363,1.102642,1.974026,0.112860,1.373283,2022-01-03,2022-12-31
6,test,2022,3,Dynamic panel FE,216476,598,362,0.982944,1.564954,0.216758,1.358054,2022-01-04,2022-12-31
7,test,2022,3,Persistence: future = today,216476,598,362,1.065467,1.845093,0.041455,1.358054,2022-01-04,2022-12-31
8,test,2022,3,Seasonal naive: same weekday last week,216476,598,362,1.092006,1.925490,0.102168,1.358054,2022-01-04,2022-12-31
9,test,2023,1,Dynamic panel FE,217074,598,363,0.219105,0.494748,0.071986,0.167473,2023-01-01,2023-12-29


Results saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\dynamic_panel_nj_outputs\window_90d_20261024T185556193544Z


## 4. Inspect one region's predictions

`date` is the forecast origin and `target_date` is the day being predicted. A given actual target date can have separate forecasts issued 1, 2 and 3 days earlier. These are different forecasting tasks, not duplicate samples. `fit_date` identifies the most recent weekly parameter refit.


In [11]:
example_zip = predictions["zip"].iloc[0]
example = predictions.loc[
    predictions["zip"].eq(example_zip) & predictions["split"].eq("test")
].sort_values(["date","horizon"]).head(30)
display(example[["zip", "date", "target_date", "horizon", "actual_cases",
                 "panel_prediction", "persistence_prediction", "seasonal_naive_prediction"]])


,zip,date,target_date,horizon,actual_cases,panel_prediction,persistence_prediction,seasonal_naive_prediction
216476,07001,2022-01-01,2022-01-02,1,7.0,7.129923,5.0,12.0
867698,07001,2022-01-01,2022-01-03,2,4.0,7.814529,5.0,7.0
1518920,07001,2022-01-01,2022-01-04,3,1.0,7.644314,5.0,9.0
216477,07001,2022-01-02,2022-01-03,1,4.0,7.640426,7.0,7.0
867699,07001,2022-01-02,2022-01-04,2,1.0,7.186174,7.0,9.0
1518921,07001,2022-01-02,2022-01-05,3,4.0,7.241494,7.0,11.0
216478,07001,2022-01-03,2022-01-04,1,1.0,5.156348,4.0,9.0
867700,07001,2022-01-03,2022-01-05,2,4.0,5.066482,4.0,11.0
1518922,07001,2022-01-03,2022-01-06,3,4.0,4.913203,4.0,8.0
216479,07001,2022-01-04,2022-01-05,1,4.0,2.916612,1.0,11.0


## Saved files

- `experiment_config.json`: window, splits, features, source fingerprint, model, versions and limitations.
- `training_log.csv`: each refit's training origin range, latest training target, prediction block and row counts.
- `metrics_by_horizon.csv`: all-region validation/test MAE and RMSE.
- `metrics_matched_pressure_comparison.csv`: same-region/same-row pressure comparison.
- `metrics_by_year.csv` and `metrics_by_region.csv`: main baseline diagnostics.
- `feature_coefficients.csv`: shared predictive coefficients at each refit; not causal effects.
- `predictions.csv.gz`: every forecast, actual count, raw and clipped predictions, and simple baselines.
- `region_cohort.csv`: pressure-comparison eligibility.
- `prediction_preview.csv` and `run_summary.json`: compact preview and execution checks.

No outbreak metrics are populated with placeholders. PR-AUC and warning lead time require an agreed event definition; spatial residual diagnostics require agreed geographic weights. Clear real-data outputs before committing the notebook. This notebook does not upload anything.
